# Bounded panic evidence: integrated validation

Integrated commit: `62f203acc5d4e42908abb871faec66169f78f551`. The integrated tree matches reviewed source/evidence commit `11d72a0ea294766d676ce8e78dcf0717696cf482`.

This notebook runs source → typed facts → Z3 → native witness experiments, with DuckDB analysis of the results. It keeps syntax edges, compiler observations, bounded solver claims and runtime reproduction distinct. The matching regression finished **1,292 passed / 0 failed / 19 existing ignores**, including both doctests, across 98 targets.

Fresh cases vary arithmetic configuration and input domains. Unsupported cases retain unknown obligations; forged evidence must fail closed. Compiler/CGA comparisons require exact source/build provenance. Bounded UNSAT applies only to the encoded domain.

Root validation: `bd-3ez`. Previous notebooks and published graph state are preserved.

The motivating [Tweede golf article](https://tweedegolf.nl/en/blog/242/finding-panics-with-call-graph-analyzer/) describes MIR-based call-graph extraction and conservative handling of dynamic calls. Here the solver adds value/guard constraints, and native replay checks candidate witnesses.

In [2]:
from pathlib import Path
import json, hashlib, subprocess, os, datetime
REVIEW = Path("/private/tmp/r8g-root-review-worktree")
ARTIFACT = Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation")
ARTIFACT.mkdir(parents=True, exist_ok=True)
BIN = Path("/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence")
INTEGRATION = "62f203acc5d4e42908abb871faec66169f78f551"
sha = lambda b: hashlib.sha256(b).hexdigest()
assert sha(BIN.read_bytes()) == "984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd"
subprocess.run(["git", "diff", "--exit-code", INTEGRATION, "11d72a0ea294766d676ce8e78dcf0717696cf482"], cwd=REVIEW, check=True, capture_output=True)
checked = subprocess.run(["python3", str(REVIEW / "docs/research/artifacts/2026-11-01-panic-implementation/R8/final-evidence/verify.py")], cwd=REVIEW, text=True, capture_output=True, check=True)
verification = json.loads(checked.stdout)
assert verification["all_passed"]
integrity = {"integration_commit": INTEGRATION, "reviewed_commit": "11d72a0ea294766d676ce8e78dcf0717696cf482", "trees_match": True, "binary": str(BIN), "binary_sha256": sha(BIN.read_bytes()), "source_manifest_sha256": "9cf6d4cebe4754404ff5b5240a49cb3ac2baa4274934f5d8a35ccfb195d52b46", "verification": verification, "verified_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
(ARTIFACT / "integrity.json").write_text(json.dumps(integrity, indent=2) + "\n")
print(json.dumps(integrity, indent=2))

{
  "integration_commit": "62f203acc5d4e42908abb871faec66169f78f551",
  "reviewed_commit": "11d72a0ea294766d676ce8e78dcf0717696cf482",
  "trees_match": true,
  "binary": "/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence",
  "binary_sha256": "984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd",
  "source_manifest_sha256": "9cf6d4cebe4754404ff5b5240a49cb3ac2baa4274934f5d8a35ccfb195d52b46",
  "verification": {
    "all_passed": true,
    "source_inputs": 2434,
    "historical_files": 966,
    "frozen_files": 388,
    "fresh_mcp_receipts": 9,
    "cli_controls": 6,
    "native_assertion_reproductions": 2,
    "forgery_rejections": 10,
    "wrong_payload_nonreproductions": 2,
    "r8g_test_targets": 98,
    "r8g_passed": 1292,
    "r8g_failed": 0,
    "r8g_existing_ignores": 19,
    "r8g_enumerated_cases": 3584,
    "historical_receipt_availability": {
      "unavailable": 258,
      "available": 173
    }
  },
  "verified_at_utc": "2026-11-02T04:18:18.119

In [3]:
SELECT * FROM duckdb_extensions() WHERE extension_name IN ('json','parquet','httpfs','shellfs','petgraph','duckpgq','duckrouting') ORDER BY extension_name;

extension_name,loaded,installed,install_path,description,aliases,extension_version,install_mode,installed_from
duckrouting,False,True,/Users/kevintruong/.duckdb/extensions/v1.5.5/osx_arm64/duckrouting.duckdb_extension,,[],509ece1,REPOSITORY,community
httpfs,True,True,/Users/kevintruong/.duckdb/extensions/v1.5.5/osx_arm64/httpfs.duckdb_extension,Adds support for reading and writing files over a HTTP(S) connection,"[http, https, s3]",827222f,REPOSITORY,core
json,True,True,(BUILT-IN),Adds support for JSON operations,[],v1.5.5,STATICALLY_LINKED,
parquet,True,True,(BUILT-IN),Adds support for reading and writing parquet files,[],v1.5.5,STATICALLY_LINKED,
shellfs,False,True,/Users/kevintruong/.duckdb/extensions/v1.5.5/osx_arm64/shellfs.duckdb_extension,,[],4f724fc,REPOSITORY,community


In [6]:
import sys
CGA_WORK = Path("/private/tmp/r8-notebook-cga-20261102")
if "cga_job" not in globals():
    assert not CGA_WORK.exists(), "Preserve an existing CGA run; use a new explicit directory for a new run."
    cga_script = ARTIFACT / "cga-bootstrap.py"
    cga_script.write_text("from pathlib import Path\nimport os, sys, subprocess, hashlib, json, time\nWORK=Path(\"/private/tmp/r8-notebook-cga-20261102\")\nART=Path(\"/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation\")\nREVIEW=Path(\"/private/tmp/r8g-root-review-worktree\")\nREV=\"2fc63108f0b674fe7a178191ffa15647a18879f1\"\nTOOLCHAIN=\"nightly-2026-10-18\"\ndef sha(b): return hashlib.sha256(b).hexdigest()\ndef status(v): (ART/\"cga-bootstrap-status.json\").write_text(json.dumps(v,indent=2)+\"\\n\")\ndef run(name,args,cwd=None,env=None):\n    start=time.time()\n    p=subprocess.run(args,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=900)\n    (ART/(name+\".stdout\")).write_bytes(p.stdout)\n    (ART/(name+\".stderr\")).write_bytes(p.stderr)\n    (ART/(name+\".command.json\")).write_text(json.dumps({\"argv\":args,\"cwd\":str(cwd) if cwd else None,\"exit\":p.returncode,\"seconds\":time.time()-start,\"stdout_sha256\":sha(p.stdout),\"stderr_sha256\":sha(p.stderr)},indent=2)+\"\\n\")\n    if p.returncode: raise RuntimeError(name+\" exited \"+str(p.returncode)+\"; see retained stderr\")\n    return p.stdout\ntry:\n    WORK.mkdir(exist_ok=False)\n    source=WORK/\"source\"\n    run(\"cga-git-clone\",[\"git\",\"clone\",\"--filter=blob:none\",\"--no-checkout\",\"https://github.com/OpenDevicePartnership/cga.git\",str(source)])\n    run(\"cga-git-checkout\",[\"git\",\"-C\",str(source),\"checkout\",\"--detach\",REV])\n    patch=REVIEW/\"docs/research/artifacts/2026-11-01-cga-isolation/upstream/original-macos-rpath.patch\"\n    assert sha(patch.read_bytes())==\"72f3d4cd64439d988deba0ee1e0f449df8290e6f1d1e0ae83ba82d97b1592aca\"\n    run(\"cga-apply-retained-macos-patch\",[\"git\",\"-C\",str(source),\"apply\",str(patch)])\n    actual=run(\"cga-producer-patch\",[\"git\",\"-C\",str(source),\"diff\",\"--binary\",\"HEAD\"])\n    assert actual==patch.read_bytes()\n    env=os.environ.copy()\n    env.update(SPUR_REMOTE=\"0\",SPUR_SCCACHE_S3=\"0\",RUSTC_WRAPPER=\"\",CARGO_INCREMENTAL=\"0\",RUSTUP_TOOLCHAIN=TOOLCHAIN,TMPDIR=\"/private/tmp\")\n    status({\"status\":\"building\",\"source\":str(source),\"revision\":REV,\"toolchain\":TOOLCHAIN,\"retained_patch_sha256\":sha(actual)})\n    target=WORK/\"target\"\n    wrapper=\"/Users/kevintruong/project/SpurCode/scripts/spur-cargo\"\n    run(\"cga-build\",[wrapper,\"--workdir\",str(source),\"+\"+TOOLCHAIN,\"build\",\"--locked\",\"--bins\",\"--target-dir\",str(target)],cwd=source,env=env)\n    driver=target/\"debug/cga-driver\";launcher=target/\"debug/cargo-cga\"\n    assert driver.is_file() and launcher.is_file()\n    assert run(\"cga-final-source-patch\",[\"git\",\"-C\",str(source),\"diff\",\"--binary\",\"HEAD\"])==actual\n    assert not run(\"cga-source-untracked\",[\"git\",\"-C\",str(source),\"ls-files\",\"--others\",\"--exclude-standard\"])\n    status({\"status\":\"ready\",\"source\":str(source),\"driver\":str(driver),\"launcher\":str(launcher),\"revision\":REV,\"toolchain\":TOOLCHAIN,\"patch\":str(patch),\"patch_sha256\":sha(actual),\"driver_sha256\":sha(driver.read_bytes()),\"launcher_sha256\":sha(launcher.read_bytes())})\nexcept Exception as e:\n    status({\"status\":\"blocked\",\"reason\":str(e)})\n    raise\n")
    cga_bootstrap_log = (ARTIFACT / "cga-bootstrap.log").open("wb")
    cga_job = subprocess.Popen([sys.executable, str(cga_script)], stdout=cga_bootstrap_log, stderr=subprocess.STDOUT, cwd=ARTIFACT)
print(json.dumps({"bootstrap_pid": cga_job.pid, "exit": cga_job.poll(), "status_file": str(ARTIFACT / "cga-bootstrap-status.json"), "purpose": "Rebuild exact pinned CGA in a new temporary directory; replay only the retained one-line macOS link patch, leaving originals unchanged."}, indent=2))

{
  "bootstrap_pid": 43964,
  "exit": null,
  "status_file": "/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/cga-bootstrap-status.json",
  "purpose": "Rebuild exact pinned CGA in a new temporary directory; replay only the retained one-line macOS link patch, leaving originals unchanged."
}


In [7]:
case_script = ARTIFACT / "run-bounded-cases.py"
case_script.write_text("from pathlib import Path\nimport copy, datetime, hashlib, json, os, subprocess, tempfile\nBIN=Path(\"/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence\")\nART=Path(\"/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation\")\nRUN=ART/(\"bounded-\"+datetime.datetime.now(datetime.timezone.utc).strftime(\"%Y%m%dT%H%M%S%f\"))\nRUN.mkdir()\nENV=os.environ.copy();ENV.update(RUSTUP_TOOLCHAIN=\"1.94.1-aarch64-apple-darwin\",TMPDIR=\"/private/tmp\")\nsha=lambda b:hashlib.sha256(b).hexdigest()\nassert sha(BIN.read_bytes())==\"984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd\"\ndef save(path,v):path.write_text(json.dumps(v,indent=2)+\"\\n\")\ndef cli(name,req,negative=False):\n    payload=json.dumps(req).encode()\n    p=subprocess.run([str(BIN)],input=payload,stdout=subprocess.PIPE,stderr=subprocess.PIPE,cwd=RUN,env=ENV,timeout=90)\n    save(RUN/(name+\".request.json\"),req)\n    (RUN/(name+\".stdout\")).write_bytes(p.stdout);(RUN/(name+\".stderr\")).write_bytes(p.stderr)\n    save(RUN/(name+\".command.json\"),{\"binary_sha256\":sha(BIN.read_bytes()),\"input_sha256\":sha(payload),\"exit\":p.returncode,\"environment\":{\"RUSTUP_TOOLCHAIN\":ENV[\"RUSTUP_TOOLCHAIN\"],\"TMPDIR\":ENV[\"TMPDIR\"]}})\n    if negative:\n        assert p.returncode and b\"invalid panic evidence request\" not in p.stderr,(name,p.stderr)\n        return {\"rejected\":True,\"error\":p.stderr.decode().strip()}\n    assert p.returncode==0,(name,p.stderr.decode())\n    return json.loads(p.stdout)\nversion=subprocess.check_output([\"rustc\",\"-vV\"],env=ENV,text=True)\nhost=next(x[6:] for x in version.splitlines() if x.startswith(\"host: \"))\ndef build(graph,overflow):\n    text=subprocess.check_output([\"rustc\",\"--print\",\"cfg\",\"--edition\",\"2021\",\"--target\",host,\"-C\",f\"overflow-checks={str(overflow).lower()}\",\"-C\",\"debug-assertions=true\",\"-C\",\"panic=unwind\"],env=ENV,text=True)\n    atoms=[]\n    for line in text.splitlines():\n        key,sep,value=line.partition(\"=\");atoms.append({\"key\":key,\"value\":json.loads(value) if sep else None})\n    atoms.sort(key=lambda a:(a[\"key\"],a[\"value\"] or \"\"))\n    return {\"schema_version\":\"panic-build-config-v1\",\"graph_content_hash\":graph[\"graph_content_hash\"],\"target\":host,\"edition\":\"2021\",\"cfg\":atoms,\"overflow_checks\":overflow,\"debug_assertions\":True,\"panic_strategy\":\"unwind\",\"rustc_verbose\":version,\"codegen_flags\":[],\"features\":[]}\ndef run_case(name,source,domain,overflow,expected,payload=None):\n    key=name+\"-\"+str(overflow).lower()\n    with tempfile.TemporaryDirectory(dir=\"/private/tmp\") as tmp:\n        root=Path(tmp);(root/\"src\").mkdir();(root/\"src/lib.rs\").write_text(source)\n        snapshot=cli(key+\"-snapshot\",{\"command\":\"snapshot\",\"root\":tmp,\"source_paths\":[\"src/lib.rs\"]})\n    graph=snapshot[\"graph\"];sources=[{\"file_path\":\"src/lib.rs\",\"text\":source}]\n    analyzed=cli(key+\"-analyze\",{\"command\":\"analyze\",\"graph\":graph,\"sources\":sources,\"build\":build(graph,overflow)})\n    assert analyzed[\"status\"]==\"collected\",(key,analyzed)\n    facts=analyzed[\"facts\"]\n    sym=next(s for s in graph[\"symbols\"] if s[\"entity_name\"]==\"probe\")\n    body=next(b[\"body_id\"] for b in facts[\"bodies\"] if b.get(\"definition_stable_symbol_id\")==sym[\"stable_symbol_id\"])\n    parameters=[v for v in facts[\"values\"] if v[\"context\"][\"location\"][\"body_id\"]==body and v[\"origin\"][\"kind\"]==\"parameter\"]\n    request={\"command\":\"solve\",\"graph\":graph,\"sources\":sources,\"build\":build(graph,overflow),\"facts\":facts,\"domains\":[{\"body_id\":body,\"parameter_index\":v[\"origin\"][\"index\"],\"domain\":{\"kind\":\"u8\",\"minimum\":domain[0],\"maximum\":domain[1]}} for v in parameters],\"limits\":{\"entry_body_ids\":[body],\"expected_build_hash\":facts[\"build_hash\"],\"max_call_depth\":0,\"max_loop_iterations\":0,\"max_states\":4096,\"timeout_ms\":30000,\"persist\":True}}\n    report=cli(key+\"-solve\",request)[\"report\"];q=report[\"queries\"][0]\n    assert q[\"outcome\"]==expected,(key,q[\"outcome\"],q[\"obligations\"])\n    candidate=q.get(\"candidate\")\n    witness=None if not candidate or not candidate[\"inputs\"] else next(iter(candidate[\"inputs\"].values()))\n    row={\"case_id\":key,\"family\":name,\"overflow_checks\":overflow,\"minimum\":domain[0],\"maximum\":domain[1],\"graph_content_hash\":graph[\"graph_content_hash\"],\"source_oid\":facts[\"sources\"][0][\"source_oid\"],\"build_hash\":facts[\"build_hash\"],\"graph_calls\":sum(e[\"relation\"]==\"calls\" for e in graph[\"edges\"]),\"macro_sites\":len(graph[\"macro_invocations\"][\"rows\"]),\"ir_operations\":len(facts[\"extraction\"][\"operations\"]),\"raw_status\":q[\"response\"][\"status\"],\"outcome\":q[\"outcome\"],\"unknown_obligations\":len(q[\"obligations\"]),\"witness\":witness,\"runtime_reproduced\":False,\"panic_payload\":None,\"request_identity\":q[\"request_identity\"],\"solve_id\":q[\"response\"][\"solve_id\"],\"run_directory\":str(RUN),\"expected_result_matched\":True}\n    if payload is not None:\n        expected_payload=payload(witness) if callable(payload) else payload\n        replay={**copy.deepcopy(request),\"command\":\"replay\",\"report\":report,\"query_index\":0,\"expected_request_identity\":q[\"request_identity\"],\"expected_panic_payload\":expected_payload,\"execution_timeout_ms\":15000}\n        observed=cli(key+\"-replay\",replay)[\"replay\"]\n        assert observed[\"runtime_reproduced\"] and observed[\"actual_panic_payload\"]==expected_payload\n        assert candidate[\"runtime_reproduced\"] is False\n        row.update(runtime_reproduced=True,panic_payload=observed[\"actual_panic_payload\"])\n        if name==\"assertion\":\n            for field in (\"source\",\"build\",\"request\",\"model\",\"receipt\"):\n                bad=copy.deepcopy(replay)\n                if field==\"source\":bad[\"sources\"][0][\"text\"]+=\" \"\n                elif field==\"build\":bad[\"build\"][\"overflow_checks\"]=not overflow\n                elif field==\"request\":bad[\"expected_request_identity\"]=\"sha256:\"+\"0\"*64\n                elif field==\"model\":\n                    values=bad[\"report\"][\"queries\"][0][\"candidate\"][\"inputs\"];values[next(iter(values))]=0\n                else:bad[\"report\"][\"queries\"][0][\"receipt_binding\"]=\"sha256:\"+\"0\"*64\n                rejected=cli(key+\"-forged-\"+field,bad,True)\n                forgeries.append({\"case_id\":key,\"field\":field,**rejected})\n            wrong=copy.deepcopy(replay);wrong[\"expected_panic_payload\"]=\"wrong\"\n            out=cli(key+\"-wrong-payload\",wrong)[\"replay\"]\n            assert not out[\"runtime_reproduced\"] and out[\"outcome\"]==\"panic_payload_mismatch\"\n            wrong_payloads.append({\"case_id\":key,\"outcome\":out[\"outcome\"],\"actual\":out[\"actual_panic_payload\"]})\n    if expected==\"incomplete_semantics\":assert row[\"unknown_obligations\"]>0\n    return row\nrows=[];forgeries=[];wrong_payloads=[]\nassertion='pub fn probe(value:u8) { assert!(value != 173, \"Notebook witness 173\"); }\\n'\nopaque='pub fn probe(value:u8) { assert!(value != 173, \"value {}\", value); }\\n'\nfor overflow in (False,True):\n    rows.append(run_case(\"assertion\",assertion,(0,255),overflow,\"candidate_witness\",\"Notebook witness 173\"))\n    rows.append(run_case(\"restricted-domain\",assertion,(0,172),overflow,\"no_counterexample_within_complete_scope\"))\n    rows.append(run_case(\"formatted-macro\",opaque,(0,255),overflow,\"incomplete_semantics\"))\n    rows.append(run_case(\"plain-add\",'pub fn probe(value:u8) -> u8 { value + 1 }\\n',(0,255),overflow,\"candidate_witness\" if overflow else \"no_counterexample_within_complete_scope\",\"attempt to add with overflow\" if overflow else None))\n    rows.append(run_case(\"checked-add\",'pub fn probe(value:u8) -> u8 { value.checked_add(1).unwrap() }\\n',(0,255),overflow,\"candidate_witness\",\"called \\u0060Option::unwrap()\\u0060 on a \\u0060None\\u0060 value\"))\n    rows.append(run_case(\"bounds\",'pub fn probe(value:u8) -> u8 { let values: [u8;3] = [11,22,33]; values[value as usize] }\\n',(0,4),overflow,\"candidate_witness\",lambda x:f\"index out of bounds: the len is 3 but the index is {x}\"))\nrows.append(run_case(\"guarded-assertion\",'pub fn probe(value:u8) { if value < 10 { assert!(value < 11, \"guarded\"); } }\\n',(0,255),True,\"no_counterexample_within_complete_scope\"))\nsave(RUN/\"cases.json\",rows);save(RUN/\"forgeries.json\",forgeries);save(RUN/\"wrong-payloads.json\",wrong_payloads)\nsummary={\"run_directory\":str(RUN),\"cases\":len(rows),\"native_reproduced\":sum(r[\"runtime_reproduced\"] for r in rows),\"bounded_no_counterexample\":sum(r[\"outcome\"]==\"no_counterexample_within_complete_scope\" for r in rows),\"incomplete_semantics\":sum(r[\"outcome\"]==\"incomplete_semantics\" for r in rows),\"forgery_rejections\":len(forgeries),\"wrong_payload_nonreproductions\":len(wrong_payloads),\"all_expected_results_matched\":all(r[\"expected_result_matched\"] for r in rows),\"compiler\":version,\"binary_sha256\":sha(BIN.read_bytes())}\nsave(ART/\"cases.json\",rows);save(ART/\"forgeries.json\",forgeries);save(ART/\"cases-summary.json\",summary)\nprint(json.dumps(summary,indent=2),flush=True)\n")
case_result = subprocess.run([sys.executable, str(case_script)], cwd=ARTIFACT, text=True, capture_output=True, timeout=180)
(ARTIFACT / "bounded-run.stdout").write_text(case_result.stdout)
(ARTIFACT / "bounded-run.stderr").write_text(case_result.stderr)
print(case_result.stdout)
assert case_result.returncode == 0, case_result.stderr

{
  "run_directory": "/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/bounded-20261102T042551429206",
  "cases": 13,
  "native_reproduced": 7,
  "bounded_no_counterexample": 4,
  "incomplete_semantics": 2,
  "forgery_rejections": 10,
  "wrong_payload_nonreproductions": 2,
  "all_expected_results_matched": true,
  "compiler": "rustc 1.94.1 (e408947bf 2026-03-25)\nbinary: rustc\ncommit-hash: e408947bfd200af42db322daf0fadfe7e26d3bd1\ncommit-date: 2026-03-25\nhost: aarch64-apple-darwin\nrelease: 1.94.1\nLLVM version: 21.1.8\n",
  "binary_sha256": "984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd"
}



In [8]:
cga_status_path = ARTIFACT / "cga-bootstrap-status.json"
cga_status = json.loads(cga_status_path.read_text()) if cga_status_path.exists() else {"status": "starting"}
print(json.dumps({"process_exit": cga_job.poll(), **cga_status}, indent=2))
if cga_status.get("status") == "blocked":
    for log in sorted(ARTIFACT.glob("cga-*.stderr")):
        if log.stat().st_size:
            print(log.name, log.read_text(errors="replace")[-1600:])

{
  "process_exit": 0,
  "status": "ready",
  "source": "/private/tmp/r8-notebook-cga-20261102/source",
  "driver": "/private/tmp/r8-notebook-cga-20261102/target/debug/cga-driver",
  "launcher": "/private/tmp/r8-notebook-cga-20261102/target/debug/cargo-cga",
  "revision": "2fc63108f0b674fe7a178191ffa15647a18879f1",
  "toolchain": "nightly-2026-10-18",
  "patch": "/private/tmp/r8g-root-review-worktree/docs/research/artifacts/2026-11-01-cga-isolation/upstream/original-macos-rpath.patch",
  "patch_sha256": "72f3d4cd64439d988deba0ee1e0f449df8290e6f1d1e0ae83ba82d97b1592aca",
  "driver_sha256": "0251f7cdda9a4c21dee689394a999704e0c182ce51530f88e4c336afe42360b2",
  "launcher_sha256": "4d49e584f723fc7b17eafc4534352a319a96aeb045fd5ce849532cdedbf423a7"
}


In [9]:
assert cga_status["status"] == "ready", cga_status
CGA_CAPTURE = Path("/private/tmp/r8-notebook-cga-capture-20261102")
if "capture_job" not in globals():
    assert not CGA_CAPTURE.exists(), "Do not overwrite a previous compiler capture."
    capture_argv = [sys.executable, str(REVIEW / "scripts/experiments/panic-evidence-capture.py"), "--work-dir", str(CGA_CAPTURE), "--cga-source", cga_status["source"], "--cga-driver", cga_status["driver"], "--cargo-cga", cga_status["launcher"], "--producer-patch", cga_status["patch"], "--expected-driver-sha256", cga_status["driver_sha256"], "--expected-launcher-sha256", cga_status["launcher_sha256"], "--example", str(BIN)]
    (ARTIFACT / "cga-capture-invocation.json").write_text(json.dumps({"argv":capture_argv,"pinned_revision":cga_status["revision"],"driver_sha256":cga_status["driver_sha256"],"source_patch_sha256":cga_status["patch_sha256"]},indent=2)+"\n")
    capture_log = (ARTIFACT / "cga-capture.log").open("wb")
    capture_job = subprocess.Popen(capture_argv, cwd=ARTIFACT, stdout=capture_log, stderr=subprocess.STDOUT)
print(json.dumps({"capture_pid":capture_job.pid,"exit":capture_job.poll(),"capture_directory":str(CGA_CAPTURE),"configurations":5,"claim":"Fresh pinned compiler observations; failure remains failure, never a synthetic graph."},indent=2))

{
  "capture_pid": 49056,
  "exit": null,
  "capture_directory": "/private/tmp/r8-notebook-cga-capture-20261102",
  "configurations": 5,
  "claim": "Fresh pinned compiler observations; failure remains failure, never a synthetic graph."
}


In [10]:
stream_source = """from pathlib import Path
import json
p=Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/cases.json')
rows=json.loads(p.read_text())
for r in rows:
    if r['runtime_reproduced']:
        assert r['raw_status']=='sat' and r['outcome']=='candidate_witness' and r['unknown_obligations']==0
    if r['outcome']=='incomplete_semantics':
        assert r['unknown_obligations']>0
    assert r['expected_result_matched']
print(json.dumps(rows))
"""
(ARTIFACT / "emit-verified-cases.py").write_text(stream_source)
print(json.dumps({"capture_exit":capture_job.poll(),"capture_summary_exists":(CGA_CAPTURE / "summary.json").exists(),"capture_blocked_exists":(CGA_CAPTURE / "blocked.json").exists(),"stream_adapter":"emit-verified-cases.py"},indent=2))
if (CGA_CAPTURE / "summary.json").exists():
    fresh_cga=json.loads((CGA_CAPTURE / "summary.json").read_text())
    print(json.dumps({"mandatory_gate":fresh_cga["mandatory_gate"],"configurations":len(fresh_cga["configurations"]),"example_sha256":fresh_cga["example_sha256"],"first_configuration_keys":list(fresh_cga["configurations"][0])},indent=2))
elif (CGA_CAPTURE / "blocked.json").exists():
    print((CGA_CAPTURE / "blocked.json").read_text())

{
  "capture_exit": 0,
  "capture_summary_exists": true,
  "capture_blocked_exists": false,
  "stream_adapter": "emit-verified-cases.py"
}
{
  "mandatory_gate": "passed",
  "configurations": 5,
  "example_sha256": "984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd",
  "first_configuration_keys": [
    "name",
    "build_id",
    "raw_export_sha256",
    "raw_export",
    "inputs_before_sha256",
    "inputs_after_sha256",
    "runtime_controls_passed",
    "negative_controls_passed",
    "acceptance",
    "observation_count",
    "association_counts"
  ]
}


In [11]:
CREATE OR REPLACE TEMP TABLE panic_cases AS
SELECT * FROM read_json_auto('python3 /Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/emit-verified-cases.py |');
COPY panic_cases TO '/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/cases.parquet' (FORMAT PARQUET);
SELECT family, overflow_checks, graph_calls, macro_sites, ir_operations,
       raw_status, outcome, unknown_obligations, witness, runtime_reproduced
FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/cases.parquet')
ORDER BY family, overflow_checks;

family,overflow_checks,graph_calls,macro_sites,ir_operations,raw_status,outcome,unknown_obligations,witness,runtime_reproduced
assertion,False,0,1,1,sat,candidate_witness,0,173,True
assertion,True,0,1,1,sat,candidate_witness,0,173,True
bounds,False,0,0,1,sat,candidate_witness,0,3,True
bounds,True,0,0,1,sat,candidate_witness,0,3,True
checked-add,False,2,0,0,sat,candidate_witness,0,255,True
checked-add,True,2,0,0,sat,candidate_witness,0,255,True
formatted-macro,False,0,1,0,unsat,incomplete_semantics,1,<NA>,False
formatted-macro,True,0,1,0,unsat,incomplete_semantics,1,<NA>,False
guarded-assertion,True,0,1,1,unsat,no_counterexample_within_complete_scope,0,<NA>,False
plain-add,False,0,0,1,unsat,no_counterexample_within_complete_scope,0,<NA>,False


In [12]:
import shutil, collections
assert capture_job.poll() == 0, (ARTIFACT / "cga-capture.log").read_text(errors="replace")[-3000:]
fresh_cga = json.loads((CGA_CAPTURE / "summary.json").read_text())
assert fresh_cga["mandatory_gate"] == "passed" and len(fresh_cga["configurations"]) == 5
CGA_ARCHIVE = ARTIFACT / "cga-capture"
archive_rows = []
for folder, directories, files in os.walk(CGA_CAPTURE):
    directories[:] = [d for d in directories if d not in {"target", "identity-target", "cargo-home"}]
    for filename in files:
        original = Path(folder) / filename
        relative = original.relative_to(CGA_CAPTURE)
        retained = CGA_ARCHIVE / relative
        retained.parent.mkdir(parents=True, exist_ok=True)
        raw = original.read_bytes()
        if retained.exists():
            assert retained.read_bytes() == raw, str(relative)
        else:
            retained.write_bytes(raw)
        archive_rows.append({"path": str(relative), "bytes": len(raw), "sha256": sha(raw)})
(ARTIFACT / "cga-archive-manifest.json").write_text(json.dumps(archive_rows, indent=2)+"\n")
config_rows = []
for config in fresh_cga["configurations"]:
    raw_path = CGA_ARCHIVE / config["name"] / "raw-export.json"
    raw = json.loads(raw_path.read_text())
    assert sha(raw_path.read_bytes()) == config["raw_export_sha256"]
    config_rows.append({**{k:v for k,v in config.items() if k != "raw_export"}, "nodes":len(raw["nodes"]), "roots":len(raw["roots"])})
(ARTIFACT / "cga-configurations.json").write_text(json.dumps(config_rows, indent=2)+"\n")
first = fresh_cga["configurations"][0]["name"]
raw_first = json.loads((CGA_ARCHIVE / first / "raw-export.json").read_text())
import_first = json.loads((CGA_ARCHIVE / first / "import.stdout").read_text())["compiler_evidence"]
print(json.dumps({"archived_files":len(archive_rows), "archived_bytes":sum(r["bytes"] for r in archive_rows), "configurations":config_rows, "raw_graph_keys":list(raw_first), "first_node":raw_first["nodes"][0], "compiler_evidence_keys":list(import_first), "first_observation":import_first["observations"][0]},indent=2)[:10500])

{
  "archived_files": 412,
  "archived_bytes": 7907202,
  "configurations": [
    {
      "name": "d0_o0",
      "build_id": "17a0073ce818504c6cf861304d6d665906293bc23c640c245df6d7f51aa2027f",
      "raw_export_sha256": "27df1702b006375c44c6643ca7fcbf0360ab5ec6a188565fd1630e23dd8ed4cb",
      "inputs_before_sha256": "fde8eab1931a9a77f541c46271d9f8f3e63b35f0eb26d54af3c8c675b8c4242e",
      "inputs_after_sha256": "fde8eab1931a9a77f541c46271d9f8f3e63b35f0eb26d54af3c8c675b8c4242e",
      "runtime_controls_passed": 9,
      "negative_controls_passed": [
        "source",
        "config",
        "checksum",
        "reference",
        "ownership"
      ],
      "acceptance": "/private/tmp/r8-notebook-cga-capture-20261102/d0_o0/acceptance.json",
      "observation_count": 134,
      "association_counts": {
        "exact": 18,
        "unique_containing": 1,
        "ambiguous": 0,
        "unmapped": 55,
        "external": 60
      },
      "nodes": 79,
      "roots": 1
    },
    {
    

In [13]:
snapshot_cga = json.loads((CGA_ARCHIVE / "snapshot.json").read_text())
compiler_observations, compiler_nodes, compiler_summary = [], [], []
edge_serial = 1
for configuration in fresh_cga["configurations"]:
    name = configuration["name"]
    raw = json.loads((CGA_ARCHIVE / name / "raw-export.json").read_text())
    imported = json.loads((CGA_ARCHIVE / name / "import.stdout").read_text())["compiler_evidence"]
    assert imported["build_id"] == configuration["build_id"]
    counts = collections.Counter()
    for node_id, node in enumerate(raw["nodes"]):
        compiler_nodes.append({"configuration":name, "node_id":node_id, "name":node["name"], "is_root":node_id in raw["roots"]})
    for observation in imported["observations"]:
        kind = observation["kind"]
        if isinstance(kind, dict):
            assert len(kind) == 1
            tag, target = next(iter(kind.items()))
            assert tag in {"Called", "VTableCalled", "FnPtrCast"} and 0 <= target < len(raw["nodes"])
        else:
            tag, target = kind, None
            assert tag in {"BuiltinAssert", "AsyncPanic"}
        source = observation["from_node"]
        assert 0 <= source < len(raw["nodes"])
        assert observation["canonical_caller"] == raw["nodes"][source]["name"]
        counts[tag] += 1
        compiler_observations.append({"id":edge_serial, "configuration":name, "source":source, "target":target, "kind":tag, "caller":observation["canonical_caller"], "callee":observation["canonical_callee"], "association":observation["association"]["state"], "site_id":observation["association"].get("site_id"), "build_id":imported["build_id"], "cost":1.0, "reverse_cost":-1.0})
        edge_serial += 1
    compiler_summary.append({"configuration":name, "build_id":imported["build_id"], "source_graph_hash":snapshot_cga["graph"]["graph_content_hash"], "source_graph_calls":sum(e["relation"]=="calls" for e in snapshot_cga["graph"]["edges"]), "compiler_nodes":len(raw["nodes"]), "observations":len(imported["observations"]), "exact_associations":configuration["association_counts"]["exact"], "containing_associations":configuration["association_counts"]["unique_containing"], "unmapped":configuration["association_counts"]["unmapped"], "external":configuration["association_counts"]["external"], "direct_calls":counts["Called"], "virtual_candidates":counts["VTableCalled"], "address_flow":counts["FnPtrCast"], "builtin_checks":counts["BuiltinAssert"], "native_controls":configuration["runtime_controls_passed"]})
for filename, value in [("cga-observations.json",compiler_observations),("cga-nodes.json",compiler_nodes),("cga-summary.json",compiler_summary)]:
    (ARTIFACT / filename).write_text(json.dumps(value,indent=2)+"\n")
sinks=[{"node_id":i,"name":n["name"]} for i,n in enumerate(raw_first["nodes"]) if "panic_fmt" in n["name"] or "unwrap_failed" in n["name"]]
(ARTIFACT / "cga-route-candidates.json").write_text(json.dumps(sinks,indent=2)+"\n")
print(json.dumps({"observations":len(compiler_observations),"configurations":compiler_summary,"panic_sink_candidates":sinks},indent=2))

{
  "observations": 677,
  "configurations": [
    {
      "configuration": "d0_o0",
      "build_id": "17a0073ce818504c6cf861304d6d665906293bc23c640c245df6d7f51aa2027f",
      "source_graph_hash": "b6e27b54bc5a4d50833d33ff765c7169032f2688bbb12f1facde04a99e0f0560",
      "source_graph_calls": 36,
      "compiler_nodes": 79,
      "observations": 134,
      "exact_associations": 18,
      "containing_associations": 1,
      "unmapped": 55,
      "external": 60,
      "direct_calls": 133,
      "virtual_candidates": 0,
      "address_flow": 0,
      "builtin_checks": 1,
      "native_controls": 9
    },
    {
      "configuration": "d0_o1",
      "build_id": "79679c23883f3dccc948573e41b3207a75ca693e79c63129c63c67091dc5c3ce",
      "source_graph_hash": "b6e27b54bc5a4d50833d33ff765c7169032f2688bbb12f1facde04a99e0f0560",
      "source_graph_calls": 36,
      "compiler_nodes": 79,
      "observations": 135,
      "exact_associations": 18,
      "containing_associations": 1,
      "unmapped":

## What the graph and solver contribute

The same physical assertion graph is satisfiable over `0..255` with witness `173`, and has no counterexample over `0..172`. Bounds and arithmetic have **zero ordinary call edges** here, but the semantic sidecar records their implicit operations. Formatted macros remain incomplete despite raw UNSAT.

The compiler pilot below is a separate, source-bound corpus on pinned nightly Rust. Its five configurations share one physical source graph while MIR observations vary with overflow/debug/features. Counts have different units, so their ratio is not a recall metric. Routes through MIR calls are candidate paths, not proofs of a feasible panic. Function-pointer casts and virtual-call candidates must retain their own categories; this pilot has zero observations in those two categories.

In [14]:
CREATE OR REPLACE TEMP TABLE cga_summary AS SELECT * FROM read_json_auto('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/cga-summary.json');
CREATE OR REPLACE TEMP TABLE cga_edges AS SELECT * FROM read_json_auto('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/cga-observations.json');
CREATE OR REPLACE TEMP TABLE cga_nodes AS SELECT * FROM read_json_auto('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/cga-nodes.json');
SELECT configuration, source_graph_calls, compiler_nodes, direct_calls, builtin_checks,
       exact_associations, containing_associations, unmapped, external, native_controls
FROM cga_summary ORDER BY configuration;

configuration,source_graph_calls,compiler_nodes,direct_calls,builtin_checks,exact_associations,containing_associations,unmapped,external,native_controls
d0_o0,36,79,133,1,18,1,55,60,9
d0_o1,36,79,133,2,18,1,56,60,9
d1_o0,36,79,134,1,18,1,55,61,9
d1_o1,36,79,134,2,18,1,56,61,9
d1_o1_extra,36,79,135,2,19,1,56,61,9


In [16]:
-- duckrouting prepares the edges SQL on a separate connection; read the
-- retained JSON directly instead of relying on connection-local TEMP objects.
CREATE OR REPLACE TEMP TABLE candidate_panic_route AS
SELECT * FROM duckrouting_dijkstra(
  'SELECT id::BIGINT AS id, source::BIGINT AS source, target::BIGINT AS target,
          cost::DOUBLE AS cost, reverse_cost::DOUBLE AS reverse_cost
   FROM read_json_auto(''/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/cga-observations.json'')
   WHERE configuration=''d1_o1'' AND kind=''Called'' AND target IS NOT NULL',
   0::BIGINT, 66::BIGINT);
SELECT p.path_seq, p.node, n.name AS compiler_function, p.edge, p.agg_cost
FROM candidate_panic_route p JOIN cga_nodes n ON n.configuration='d1_o1' AND n.node_id=p.node
ORDER BY p.path_seq;

path_seq,node,compiler_function,edge,agg_cost
1,0,panic_pilot::main,420,0.0
2,12,panic_pilot::generic::<u16>,480,1.0
3,76,std::option::Option::<u16>::unwrap,540,2.0
4,66,std::option::unwrap_failed,-1,3.0


In [17]:
# Cross-check the exact source-bound queries through the separate MCP solver endpoint.
# These receipt IDs supplement the original CLI-local receipts; they do not replace them.
crosscheck = json.loads((ARTIFACT / "mcp-solver-crosscheck.json").read_text())
def tool_json(value):
    return value.get("structuredContent") or json.loads(next(c["text"] for c in value["content"] if c["type"] == "text"))
run_dir = Path(json.loads((ARTIFACT / "cases-summary.json").read_text())["run_directory"])
mcp_rows = []
for saved in crosscheck["queries"]:
    report = json.loads((run_dir / (saved["case_id"] + "-solve.stdout")).read_text())["report"]
    query = report["queries"][0]
    receipt = tool_json(saved["reload"])
    response = tool_json(saved["response"])
    assert tool_json(saved["preflight"])["valid"]
    assert query["request"] == saved["request"] == receipt["request"]
    assert query["request_identity"] == saved["request_identity"]
    assert report["sources"] == saved["sources"]
    assert report["build_hash"] == saved["build_hash"]
    assert report["fact_digest"] == saved["fact_digest"]
    assert query["outcome"] == saved["analysis_outcome"]
    assert query["obligations"] == saved["unknown_obligations"]
    assert receipt["status"] == response["status"] == query["response"]["status"]
    assert receipt["solve_id"] == response["solve_id"]
    assert receipt.get("model") == response.get("model")
    if receipt["status"] == "sat":
        assert receipt["model"] == query["response"]["model"]
    if saved["analysis_outcome"] == "incomplete_semantics":
        assert saved["unknown_obligations"]
    mcp_rows.append({
        "case_id": saved["case_id"], "status": receipt["status"],
        "outcome": saved["analysis_outcome"], "solve_id": receipt["solve_id"],
        "source_request_binding_verified": True,
        "unknown_obligations": len(saved["unknown_obligations"])
    })
assert len(mcp_rows) == 5
(ARTIFACT / "mcp-crosscheck-summary.json").write_text(json.dumps(mcp_rows, indent=2) + "\n")
print(json.dumps(mcp_rows, indent=2))


[
  {
    "case_id": "assertion-true",
    "status": "sat",
    "outcome": "candidate_witness",
    "solve_id": "sol_e17c233b4a254ec3",
    "source_request_binding_verified": true,
    "unknown_obligations": 0
  },
  {
    "case_id": "restricted-domain-true",
    "status": "unsat",
    "outcome": "no_counterexample_within_complete_scope",
    "solve_id": "sol_066ad77a61944e58",
    "source_request_binding_verified": true,
    "unknown_obligations": 0
  },
  {
    "case_id": "plain-add-true",
    "status": "sat",
    "outcome": "candidate_witness",
    "solve_id": "sol_b411b82d865e4b13",
    "source_request_binding_verified": true,
    "unknown_obligations": 0
  },
  {
    "case_id": "plain-add-false",
    "status": "unsat",
    "outcome": "no_counterexample_within_complete_scope",
    "solve_id": "sol_a50c196a1f4a4f70",
    "source_request_binding_verified": true,
    "unknown_obligations": 0
  },
  {
    "case_id": "formatted-macro-true",
    "status": "unsat",
    "outcome": "incompl

In [18]:
-- The physical call graph stays the same while the domain and build change the answer.
COPY (
 SELECT family,overflow_checks,minimum,maximum,graph_content_hash,graph_calls,macro_sites,
        raw_status,outcome,unknown_obligations,witness,runtime_reproduced
 FROM panic_cases
 WHERE family IN ('assertion','restricted-domain','plain-add','formatted-macro')
 ORDER BY family,overflow_checks
) TO '/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/domain-build-comparison.json' (FORMAT JSON,ARRAY true);
COPY (
 SELECT p.path_seq,p.node,n.name AS compiler_function,p.edge,p.agg_cost
 FROM candidate_panic_route p JOIN cga_nodes n
 ON n.configuration='d1_o1' AND n.node_id=p.node ORDER BY p.path_seq
) TO '/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/candidate-panic-route.json' (FORMAT JSON,ARRAY true);
COPY (
 SELECT extension_name,extension_version,installed,loaded
 FROM duckdb_extensions()
 WHERE extension_name IN ('shellfs','duckrouting','json','parquet','httpfs')
 ORDER BY extension_name
) TO '/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/duckdb-runtime.json' (FORMAT JSON,ARRAY true);
SELECT count(*) AS cases,
       count(*) FILTER (WHERE runtime_reproduced) AS native_reproductions,
       count(*) FILTER (WHERE outcome='no_counterexample_within_complete_scope') AS bounded_no_counterexample,
       count(*) FILTER (WHERE outcome='incomplete_semantics') AS incomplete,
       count(*) FILTER (WHERE expected_result_matched) AS expected_matches
FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-panic-evidence-validation/cases.parquet');

cases,native_reproductions,bounded_no_counterexample,incomplete,expected_matches
13,7,4,2,13


## What the experiment establishes

The implemented sidecars let `spur-graph` produce useful bounded panic results from source, including operations absent from ordinary call edges. The strongest evidence is the full chain **exact source/build → typed facts and guards → Z3 query → validated model → native fixture replay**.

| Controlled comparison | Observed result | Consequence |
|---|---|---|
| Same assertion source and graph; `u8` domain 0–255 vs 0–172 | Witness `173` reproduces; restricted domain has no counterexample within complete scope | Domains are part of the claim. Graph topology alone cannot determine the answer. |
| Same `u8 + 1` source and graph; overflow checks on vs off | Witness `255` reproduces with checks; wrapping execution has no bounded panic witness | Compiler configuration must bind the arithmetic model. |
| Array indexing with zero ordinary call edges | Index `3` reproduces a bounds panic for length 3 in both modes | Explicit implicit-check semantics close a real call-graph gap. |
| `checked_add(1).unwrap()` | Witness `255` reproduces in both modes | Value origin and Option state matter beyond the call name. |
| Guarded assertion | No counterexample within the supported complete scope | Branch guards prevent a false panic witness in this fixture. |
| Formatted assertion outside the supported macro subset | Raw `UNSAT`, with an unresolved obligation and `incomplete_semantics` | An empty modeled panic set is never promoted to a safety result. |

Across 13 fresh cases: **7 native reproductions, 4 bounded no-counterexample results, and 2 explicitly incomplete results**. Ten source/build/request/model/receipt forgeries were rejected; two deliberately wrong panic payloads failed reproduction. Five exact generated requests were independently preflighted, solved by MCP Z3, persisted, and reloaded. Those are separate endpoint receipts, not replacements for CLI-local IDs.

## How CGA complements this

The fresh pinned CGA producer captured five compiler configurations successfully: **677 observations, 45 native controls, and 26 corruption rejections**. JSON joins preserve source/configuration identity and observation kind. Shellfs streams the verified case data, Parquet retains the measured table, and Duckrouting finds this three-edge candidate path:

`panic_pilot::main → panic_pilot::generic::<u16> → Option::<u16>::unwrap → std::option::unwrap_failed`

This is a compiler call-path candidate. The routing query neither reconstructs branch conditions nor proves that an input reaches the sink. Compiler node/edge counts and physical source Calls have different grains; their ratio is not recall.

## Remaining gaps and the appropriate next approach

| Gap | Appropriate evidence |
|---|---|
| Traits, autoref, macro expansions and cross-crate dispatch | Keep unresolved source obligations; add compiler-backed, exact body/callsite associations before granting a target. |
| Types or effects beyond the supported `u8`/Bool/Option subset | Extend the typed IR with source/compiler-backed semantics and differential native controls before producing complete results. |
| Larger loops, recursion or call depth | State explicit bounds and budgets; exhausted exploration stays incomplete. |
| Whole-CLI execution graph | Isolate/fix the upstream generic virtual-call drop-glue ICE before claiming a `spur::main` capture. The successful fixture entry here is `panic_pilot::main`. |
| Function-pointer and virtual candidate coverage | Add dedicated captures. This fresh pilot has zero `FnPtrCast` and zero `VTableCalled` observations; address flow must not become an execution edge. |

The stable-1.94.1 bounded solver corpus and nightly-2026-10-18 CGA corpus are separate, explicitly identified experiments. There is no claim that the solver has complete semantics for the CGA corpus, or that these fixtures prove an arbitrary workspace safe.

The retained implementation regression is source-matched: **1,292 passed, 0 failed, 19 existing ignores across 98 targets**. This notebook adds fresh experimental evidence; it does not replace that regression.


In [19]:
# Read-only validation of saved bindings and retained payload hashes.
verification_run = subprocess.run(
    [sys.executable, str(ARTIFACT / "verify.py")],
    capture_output=True, text=True, timeout=60, check=False,
)
assert verification_run.returncode == 0, verification_run.stderr
final_verification = json.loads(verification_run.stdout)
assert final_verification["all_saved_checks_passed"]
(ARTIFACT / "verification.json").write_text(json.dumps(final_verification, indent=2) + "\n")
print(json.dumps(final_verification, indent=2))


{
  "all_saved_checks_passed": true,
  "source_bound_cases": 13,
  "native_panic_reproductions": 7,
  "bounded_no_counterexamples": 4,
  "explicit_incomplete_results": 2,
  "forgery_rejections": 10,
  "wrong_payload_nonreproductions": 2,
  "separate_mcp_reloaded_receipts": 5,
  "cga_configurations": 5,
  "cga_observations": 677,
  "cga_native_controls": 45,
  "cga_corruption_rejections": 26,
  "archived_cga_files": 412,
  "candidate_route_edges": 3,
  "retained_full_regression_passed": 1292,
  "retained_full_regression_ignored": 19,
  "sealed_payloads_verified": 697,
  "scope": "Saved observation integrity and exact bindings; bounded fixture results only."
}


In [20]:
# SPUR DuckDB extension setup cell v1
# requirements: {"active":[{"packageName":"duckrouting","source":"community"},{"packageName":"shellfs","source":"community"}],"pending":[]}
# Managed by Jute. Change packages through the extension sidebar.
# SPUR DuckDB bootstrap
import duckdb
import sys
_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
# Shared by notebook setup and classic SQL cells. Preserve the generic path for
# older bundles; new bundles carry exact native ABI artifacts per engine.
from pathlib import Path as _SpurExtensionPath

_spur_extension = _SpurExtensionPath(_SPUR_DUCKDB_EXTENSION_PATH)
_spur_versioned_extension = (
    _spur_extension.parent
    / ("duckdb-v" + duckdb.__version__)
    / _spur_extension.name
)
if _spur_versioned_extension.is_file():
    _SPUR_DUCKDB_EXTENSION_PATH = str(_spur_versioned_extension)

if (
    sys.platform in ("linux", "darwin")
    and duckdb.__version__ in ("1.4.4", "1.5.5")
    and "_SPUR_DUCKDB_HOST_SYMBOLS" not in globals()
):
    import ctypes as _spur_ctypes
    import _duckdb as _spur_host_duckdb

    # Promote the already-loaded Python engine's symbols. This opens the same
    # module handle; it neither imports another engine nor replaces connections.
    _SPUR_DUCKDB_HOST_SYMBOLS = _spur_ctypes.CDLL(
        _spur_host_duckdb.__file__, mode=_spur_ctypes.RTLD_GLOBAL
    )

_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true",
                 "autoload_known_extensions": "false",
                 "autoinstall_known_extensions": "false"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
# Embedded helper source contains no token, credential path or capability lease.
# The approved local launcher passes a private inherited socket descriptor, not
# executable source or a notebook/widget message. Same-kernel code is trusted.
def _spur_managed_connection(db):
    """Only the host's selected connection is subject to managed policy."""
    import types
    connection = db.default_connection() if isinstance(db, types.ModuleType) else db
    if connection is not globals().get("_SPUR_DUCKDB_CONNECTION"):
        return None
    try:
        connection.sql("SET autoload_known_extensions = false")
        connection.sql("SET autoinstall_known_extensions = false")
    except Exception:
        return None  # existing locked/custom connections have an explicit boundary
    return connection


# Required SpurLab baseline: JSON/parquet are statically linked in the managed
# wheel; httpfs supplies remote files and REST gateway dependencies; spur_rest is
# the separately packaged, host-verified gateway (never a repository download).
# Remote datasource attach code explicitly installs/loads its provider adapter.
# Spatial and Jev are optional. No browsing/query path calls operate("install").
# Fixed metadata query only. Identifiers and descriptions remain inert JSON; no
# catalog value is interpolated as SQL. The limits bound the shell reply too.
def _spur_function_catalog(db):
    import json
    limit = 10000
    rows = db.sql("SELECT database_name, schema_name, function_name, function_type, "
                  "parameters, parameter_types, varargs, return_type, description, examples, function_oid "
                  "FROM duckdb_functions() ORDER BY database_name, schema_name, function_name, "
                  "function_type, parameter_types, varargs LIMIT 10001").fetchall()
    functions, size = [], 0
    complete = len(rows) <= limit
    for row in rows[:limit]:
        value = dict(zip(("database_name", "schema_name", "name", "function_type", "parameters",
                          "parameter_types", "varargs", "return_type", "description", "examples", "function_oid"), row))
        value["ownership"] = "unknown"
        encoded = json.dumps(value, ensure_ascii=True)
        # Oversized metadata is omitted, with incomplete coverage made explicit.
        if len(encoded) > 65536 or size + len(encoded) > 4 * 1024 * 1024:
            complete = False
            continue
        functions.append(value)
        size += len(encoded)
    return {"status": "complete" if complete else "incomplete", "functions": functions,
            "truncated": not complete}


def _spur_function_identity(value):
    return (value["database_name"], value["schema_name"], value["name"],
            value["function_type"], tuple(value["parameter_types"]), value["varargs"])


def _spur_snapshot_function_registration(db):
    try:
        snapshot = _spur_function_catalog(db)
        if snapshot["status"] == "complete" and snapshot.get("truncated") is False:
            return snapshot
    except Exception:
        pass  # Optional metadata must not prevent an authorized extension load.
    return None


def _spur_record_function_ownership(db, extension, before):
    """Retain registrations observed around a verified load on this connection."""
    if before is None:
        return
    try:
        if before["status"] != "complete" or before.get("truncated") is not False:
            return
        after = _spur_snapshot_function_registration(db)
        if after is None:
            return
        def key(row):
            return (*_spur_function_identity(row), row.get("return_type"), row["function_oid"])
        previous = {key(row) for row in before["functions"]}
        registrations = globals().setdefault("_SPUR_DUCKDB_FUNCTION_OWNERSHIP", {})
        retained = registrations.get(id(db))
        owners = dict(retained[1]) if retained is not None and retained[0] is db else {}
        additions = [key(row) for row in after["functions"]
                     if row["function_oid"] is not None and key(row) not in previous]
        for identity in additions:
            owners[identity] = set(owners.get(identity, ())) | {extension}
        registrations[id(db)] = (db, owners)
    except Exception:
        pass  # Incomplete/failed evidence stays unattributed.


class _SpurDuckDBRuntime:
    BASELINE = ("json", "parquet", "httpfs", "spur_rest")
    REPOSITORIES = {
        # DuckDB's built-in installer handles HTTP without httpfs; HTTPS itself
        # requires httpfs. Use the official signed distribution bootstrap, never
        # temporarily re-enable autoload to download its own dependency.
        "httpfs": "http://extensions.duckdb.org",
        "spatial": "http://extensions.duckdb.org",
        "jev": "http://community-extensions.duckdb.org",
    }

    def __init__(self, db, platform):
        self.db, self.platform = db, platform
        self.entries = {}
        self.inventories = {}  # inert metadata; never activation authority
        self.verified = {}  # at most one private verified copy per curated entry
        self.loaded = set()  # artifact verification is not selected-connection load

    def _entry(self, name):
        return self.entries.setdefault(name, {
            "name": name, "installed": None,
            "desired_enabled": name in self.BASELINE, "loaded": None,
            "provider_health": "unknown", "restart_required": False,
            "activation_transport_restart": False,
            "stage": "idle", "reason": None, "policy": "unknown",
            "artifact_digest": None, "signature_verified": False,
        })

    def _observe(self, db, name):
        return db.sql(
            "SELECT loaded, installed, install_path, installed_from, install_mode "
            "FROM duckdb_extensions() WHERE extension_name = ?", params=[name]
        ).fetchone()

    def _refresh(self, entry):
        row = self._observe(self.db, entry["name"])
        entry["loaded"] = bool(row and row[0])
        entry["installed"] = bool(row and row[1])
        return row

    def _artifact(self, name, install, cancelled):
        # Verification happens in a separate database with signature and ABI
        # checks ON, even when the selected gateway connection allows unsigned
        # bundled code. A repository label or hash alone is not signing evidence.
        import duckdb
        import hashlib
        from pathlib import Path
        import shutil
        import tempfile
        directory = self.db.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        config = {"allow_unsigned_extensions": "false",
                  "allow_community_extensions": "true" if name == "jev" else "false",
                  "allow_extensions_metadata_mismatch": "false",
                  "autoload_known_extensions": "false",
                  "autoinstall_known_extensions": "false"}
        config["home_directory"] = self.db.sql("SELECT current_setting('home_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        entry = self._entry(name)
        entry["signature_verified"] = False
        with duckdb.connect(config=config) as verifier:
            if (verifier.sql("SELECT version()").fetchone()[0] != "v1.5.5"
                    or verifier.sql("PRAGMA platform").fetchone()[0] != self.platform):
                raise ValueError("runtime_mismatch")
            entry["stage"] = "installing" if install else "verifying"
            row = self._observe(verifier, name)
            if install and (row is None or not row[1]):
                # Install only when absent. DuckDB rejects INSTALL across HTTP/
                # HTTPS origin spellings even for the same official repository.
                # Cached artifacts follow the same origin/signature/ABI checks
                # below; never FORCE/UPDATE or silently replace rejected bytes.
                verifier.install_extension(name, repository_url=self.REPOSITORIES[name])
                row = self._observe(verifier, name)
            if row is None or not row[1]:
                raise ValueError("not_installed")
            entry["installed"] = True
            if cancelled():
                raise InterruptedError()
            entry["stage"] = "verifying"
            repository = self.REPOSITORIES[name]
            aliases = (repository, repository.replace("http:", "https:"),
                       "community" if name == "jev" else "core")
            if row[3] not in aliases or row[4] != "REPOSITORY":
                raise ValueError("repository_mismatch")
            # Freeze the bytes in a private directory before the verifier loads
            # them. Load those same bytes into the selected connection, never a
            # mutable cache alias. Same-user/kernel arbitrary code is trusted.
            private = tempfile.TemporaryDirectory(prefix="spur-duckdb-verified-")
            try:
                path = Path(private.name) / (name + ".duckdb_extension")
                shutil.copyfile(row[2], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                try:
                    before = _spur_function_catalog(verifier)
                except Exception:
                    before = None
                verifier.load_extension(str(path))  # engine checks signature/ABI
                if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
                    raise ValueError("artifact_changed")
                try:
                    after = _spur_function_catalog(verifier)
                    if before is None:
                        raise ValueError("inventory_failed")
                    previous_rows = {_spur_function_identity(f): f for f in before["functions"]}
                    functions = [f for f in after["functions"]
                                 if previous_rows.get(_spur_function_identity(f)) != f]
                    complete = before["status"] == after["status"] == "complete"
                    inventory = {"status": "complete" if complete else "incomplete",
                                 "functions": functions, "truncated": not complete}
                except Exception:
                    inventory = {"status": "failed", "functions": [], "truncated": False}
                inventory.update(artifact_digest=digest, provenance="isolated_signed_verifier_load_delta")
                self.inventories[name] = inventory
            except BaseException:
                private.cleanup()
                raise
            previous = self.verified.get(name)
            if previous is not None:
                previous.cleanup()
            self.verified[name] = private
            return str(path), digest

    def operate(self, name, action, cancelled=lambda: False):
        """Explicit, nonsecret job; called only on the kernel shell queue.

        Cancellation is cooperative at stages. It never interrupts a user query;
        installed but not loaded artifacts remain available for an explicit retry.
        There is no UNLOAD in DuckDB: disable reports loaded + restart_required.
        """
        entry = self._entry(name)
        entry["stage"], entry["reason"] = "resolving", None
        if action == "disable" and name not in self.BASELINE:
            entry["desired_enabled"] = False
            if name == "jev":
                _spur_close_jev_channel()
        try:
            if name not in (*self.BASELINE, "spatial", "jev"):
                raise ValueError("unsupported_extension")
            if self.db is None:
                raise ValueError("no_connection")
            if self.db is not globals().get("_SPUR_DUCKDB_CONNECTION"):
                entry["policy"] = "unenforceable"
                raise ValueError("unmanaged_connection")
            if (self.db.sql("SELECT version()").fetchone()[0] != "v1.5.5"
                    or self.db.sql("PRAGMA platform").fetchone()[0] != self.platform):
                raise ValueError("runtime_mismatch")
            if _spur_managed_connection(self.db) is None:
                entry["policy"] = "unenforceable"
                raise ValueError("policy_unenforceable")
            entry["policy"] = "enforced"
            self._refresh(entry)
            if action == "disable":
                if name in self.BASELINE:
                    raise ValueError("required_baseline")
                entry["desired_enabled"] = False
                entry["restart_required"] = entry["loaded"] or name == "jev"
                entry["activation_transport_restart"] = name == "jev"
                entry["reason"] = "restart_required" if entry["restart_required"] else None
                entry["stage"] = "succeeded"
                return dict(entry)
            if cancelled():
                raise InterruptedError()
            if action == "invalidate_credential_channel" and name == "jev":
                _spur_close_jev_channel()
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
            elif action == "enable":
                entry["desired_enabled"] = True
                if name == "jev" and not self._channel_ready():
                    entry["restart_required"] = True
                    entry["activation_transport_restart"] = True
                    entry["reason"] = self._channel_reason()
            elif action in ("install", "revalidate", "load", "prepare_verification"):
                if action == "prepare_verification" and (name not in self.REPOSITORIES or not entry["installed"]
                        or not entry["signature_verified"] or not entry["artifact_digest"]):
                    raise ValueError("not_installed")
                if action == "load" and not entry["desired_enabled"]:
                    raise ValueError("setup_required_without_autoload")
                if name in ("json", "parquet"):
                    row = self._observe(self.db, name)
                    if not row or row[4] != "STATICALLY_LINKED" or not row[0]:
                        raise ValueError("baseline_unavailable")
                elif name == "spur_rest":
                    # Owned by the bundled artifact installer/bootstrap. Never
                    # weaken signature policy or download a substitute here.
                    if not entry["loaded"]:
                        raise ValueError("baseline_unavailable")
                else:
                    if entry["loaded"] and name == "httpfs" and name not in self.loaded:
                        # Connection setup may have loaded the same signed baseline
                        # first. Accept only its exact-connection frozen-byte proof.
                        from pathlib import Path
                        import hashlib
                        proof = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {}).get((id(self.db), name))
                        if proof and proof[0] is self.db and proof[1] == "core":
                            try:
                                digest = hashlib.sha256((Path(proof[2].name) / "httpfs.duckdb_extension").read_bytes()).digest()
                                if digest == proof[3]:
                                    self.verified[name] = proof[2]
                                    self.loaded.add(name)
                                    entry.update(signature_verified=True, artifact_digest=digest.hex(),
                                                 restart_required=False, activation_transport_restart=False)
                            except OSError:
                                pass
                    if entry["loaded"] and name not in self.loaded:
                        entry["signature_verified"] = False
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("preexisting_load_unverified")
                    if entry["loaded"] and action in ("load", "prepare_verification"):
                        # Explicit retry may reacquire discovery in the same isolated
                        # signature verifier; it must not change selected enablement.
                        if action == "prepare_verification" and self.inventories.get(name, {}).get("status") != "complete":
                            loaded_digest = entry["artifact_digest"]
                            _, refreshed_digest = self._artifact(name, False, cancelled)
                            if refreshed_digest != loaded_digest:
                                entry["restart_required"] = True
                                entry["activation_transport_restart"] = False
                                raise ValueError("artifact_changed")
                            entry["signature_verified"] = True
                        if name == "jev" and action == "load" and not self._channel_ready():
                            entry["restart_required"] = True
                            entry["activation_transport_restart"] = True
                            entry["reason"] = self._channel_reason()
                        if entry["restart_required"] and entry["activation_transport_restart"]:
                            entry["reason"] = self._channel_reason()
                        entry["stage"] = "succeeded"
                        return dict(entry)
                    if entry["loaded"]:
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("restart_required")
                    path, digest = self._artifact(name, action == "install", cancelled)
                    entry["installed"] = True
                    entry["artifact_digest"] = digest
                    entry["signature_verified"] = True
                    if cancelled():
                        raise InterruptedError()
                    if action in ("load", "prepare_verification") and not entry["loaded"]:
                        entry["stage"] = "loading"
                        before = _spur_snapshot_function_registration(self.db)
                        self.db.load_extension(path)
                        self._refresh(entry)
                        if not entry["loaded"]:
                            raise ValueError("load_failed")
                        self.loaded.add(name)
                        _spur_record_function_ownership(self.db, name, before)
                    if name == "jev" and action != "prepare_verification" and not self._channel_ready():
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = True
                        entry["reason"] = self._channel_reason()
            elif action != "inspect":
                raise ValueError("unsupported_operation")
            if entry["restart_required"] and entry["activation_transport_restart"]:
                entry["reason"] = self._channel_reason()
            entry["stage"] = "succeeded"
        except (KeyboardInterrupt, InterruptedError):
            entry["reason"] = "cancelled"
        except ValueError as error:
            # Only our fixed reasons are exposed; never native exception text.
            allowed = {"runtime_mismatch", "unsupported_extension", "unmanaged_connection",
                       "required_baseline", "setup_required_without_autoload", "not_installed",
                       "repository_mismatch", "artifact_changed", "baseline_unavailable",
                       "preexisting_load_unverified", "load_failed", "unsupported_operation", "no_connection",
                       "policy_unenforceable", "restart_required"}
            entry["reason"] = str(error) if str(error) in allowed else "verification_failed"
        except Exception:
            entry["reason"] = {"installing": "install_failed", "loading": "load_failed",
                               "verifying": "verification_failed"}.get(entry["stage"], "policy_failed")
        return dict(entry)

    @staticmethod
    def _channel_ready():
        import os
        if os.name != "posix":
            return False
        try:
            channel = _spur_jev_channel()
            return channel is not None and channel.available()
        except Exception:
            return False

    @staticmethod
    def _channel_reason():
        import os
        return "restart_required" if os.name == "posix" else "credential_transport_unavailable"

    def close(self):
        """Release verified temporary copies after closing this connection."""
        for directory in self.verified.values():
            directory.cleanup()
        self.verified.clear()


def _spur_duckdb_job(platform, name, action, cancel_path=None):
    """Entry point for typed host jobs; no caller SQL and no credential payload."""
    from pathlib import Path
    global _SPUR_DUCKDB_RUNTIME
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is None or current.db is not db or current.platform != platform:
        current = _SPUR_DUCKDB_RUNTIME = _SpurDuckDBRuntime(db, platform)
    return current.operate(name, action, lambda: bool(cancel_path and Path(cancel_path).exists()))


def _spur_duckdb_baseline():
    """Load approved cached baseline only; missing artifacts need explicit setup.

    Used by both root bootstrap and the library's shared SQL session helper.
    The bundled gateway loader stays with its existing host-owned installer.
    """
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    if db is None or db.sql("SELECT version()").fetchone()[0] != "v1.5.5":
        return
    platform = db.sql("PRAGMA platform").fetchone()[0]
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is not None and current.db is not db:
        current = None
    for name in ("json", "parquet", "httpfs"):
        if current is None or not current._entry(name)["loaded"]:
            _spur_duckdb_job(platform, name, "load")


class _SpurJevChannel:
    def __init__(self, descriptor):
        import socket
        self.socket = socket.socket(fileno=int(descriptor))
        self.socket.set_inheritable(False)
        self.lease = self._frame()

    def _read(self, count):
        chunks = bytearray()
        while len(chunks) < count:
            chunk = self.socket.recv(count - len(chunks))
            if not chunk:
                raise RuntimeError("Jev private credential channel closed")
            chunks.extend(chunk)
        return bytes(chunks)

    def _frame(self):
        return self._read(int.from_bytes(self._read(4), "big"))

    def _authorize(self, resolve):
        self.socket.sendall(len(self.lease).to_bytes(4, "big") + self.lease + bytes([resolve]))
        status = self._read(1)
        if status == b"\x02":
            return False
        if status != b"\x01":
            raise RuntimeError("denied")
        return True

    def available(self):
        try:
            return self._authorize(False)
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None

    def request(self):
        try:
            if not self._authorize(True):
                return None
            token = self._frame().decode("ascii")
            if not token or any(not 33 <= ord(c) <= 126 for c in token):
                raise ValueError("invalid credential")
            return token
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None


# Repeated generated SQL preambles preserve the captured channel. Failed
# attachment remains failed until kernel restart; never retry a legacy file.
if "_SPUR_JEV_BROKER" not in globals():
    _SPUR_JEV_BROKER = None
    _SPUR_JEV_BROKER_FAILED = False


def _spur_jev_channel():
    import os
    global _SPUR_JEV_BROKER, _SPUR_JEV_BROKER_FAILED
    if _SPUR_JEV_BROKER_FAILED:
        raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.")
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            if _SPUR_JEV_BROKER is not None:
                raise RuntimeError("duplicate channel")
            _SPUR_JEV_BROKER = _SpurJevChannel(descriptor)
        except Exception:
            _SPUR_JEV_BROKER_FAILED = True
            raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.") from None
    return _SPUR_JEV_BROKER


def _spur_close_jev_channel():
    """Disable/revoke/replacement is terminal for this inherited descriptor."""
    import os
    global _SPUR_JEV_BROKER_FAILED
    _SPUR_JEV_BROKER_FAILED = True
    if _SPUR_JEV_BROKER is not None:
        _SPUR_JEV_BROKER.socket.close()
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            os.close(int(descriptor))
        except (ValueError, OSError):
            pass


class _SpurJevSession:
    def __init__(self, db):
        self.db = db
        self.token = None
        self.previous = None

    def __enter__(self):
        import os
        _spur_duckdb_baseline()
        runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
        connection = _spur_managed_connection(self.db)
        if runtime is None or runtime.db is not connection:
            return self
        entry = runtime._entry("jev")
        if not (entry["desired_enabled"] and entry["loaded"] and entry["signature_verified"]
                and "jev" in runtime.loaded):
            return self
        try:
            channel = _spur_jev_channel()
            if channel is None or not channel.available():
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
                return self
        except Exception:
            # A dead/revoked channel cannot renew in-place. Do not block ordinary
            # SQL, read a legacy file or install/load anything as a fallback.
            entry["restart_required"] = True
            entry["activation_transport_restart"] = True
            entry["reason"] = "restart_required"
            return self
        endpoint = self.db.sql("SELECT current_setting('jev_api_url')").fetchone()[0]
        if endpoint != "https://api.typesafe.ai/v1/systemone":
            raise RuntimeError("Managed Jev credentials require the TypeSafe HTTPS endpoint. Reset jev_api_url before running this cell.")
        self.db.sql("RESET jev_api_key")
        # Resolve only after installation and exact endpoint validation. The
        # broker rechecks lease, enablement and generation on every request.
        try:
            token = channel.request()
        except Exception:
            entry["restart_required"] = True
            entry["reason"] = "restart_required"
            return self
        if token is None:  # disabled while setup was in flight
            return self
        self.previous = os.environ.get("TYPESAFE_API_KEY")
        self.token = token
        os.environ["TYPESAFE_API_KEY"] = token
        return self

    def __exit__(self, error_type, error, traceback):
        import os
        import traceback as traceback_module

        if self.token is None:
            return False
        token = self.token
        self.token = None
        if self.previous is None:
            os.environ.pop("TYPESAFE_API_KEY", None)
        else:
            os.environ["TYPESAFE_API_KEY"] = self.previous
        self.previous = None
        # A remote API can echo a header in an error. Suppress the original
        # traceback chain and remove the exact managed token from the diagnostic.
        if error is not None and token in "".join(traceback_module.format_exception(error_type, error, traceback)):
            message = str(error).replace(token, "[redacted]")
            token = None
            error = None
            traceback = None
            raise RuntimeError(message) from None
        return False

def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            raise ValueError("Invalid managed DuckDB extension identity")
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
            "home_directory": connection.sql("SELECT current_setting('home_directory')").fetchone()[0],
        }
        directory = connection.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchone() != connection.sql(query).fetchone():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                    verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                path = Path(private.name) / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()


_spur_duckdb_baseline()
def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            raise ValueError("Invalid managed DuckDB extension identity")
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
            "home_directory": connection.sql("SELECT current_setting('home_directory')").fetchone()[0],
        }
        directory = connection.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchone() != connection.sql(query).fetchone():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                    verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                path = Path(private.name) / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()

import json as _spur_extension_json
_spur_prepare_catalog_extensions(_SPUR_DUCKDB_CONNECTION, _spur_extension_json.loads("[{\"packageName\":\"duckrouting\",\"source\":\"community\"},{\"packageName\":\"shellfs\",\"source\":\"community\"}]"))
